In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from joblib import Parallel, delayed
from pandarallel import pandarallel
from plotly import express as px
from rdkit.Chem import PandasTools
from rdkit.Chem.rdmolfiles import MolFromSmiles, SDMolSupplier
from rdkit.rdBase import LogToPythonLogger
from rich.progress import Progress, track
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA, SparsePCA
from sklearn.manifold import TSNE
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from tqdm import tqdm
from umap import UMAP, plot


In [ ]:
from sort_and_slice_ecfp_featuriser import create_sort_and_slice_ecfp_featuriser

In [ ]:
LogToPythonLogger()
sns.set(context="notebook", style="white", rc={"figure.figsize": (14, 10)})
pandarallel.initialize(progress_bar=True)

In [ ]:
files = {
    # "DrugBank": "/homes/buttensc/Projects/semla-flow/data/unconditional/drugbank/all_structures_3d.csv",
    "DrugBank": "/homes/buttensc/Projects/semla-flow/data/unconditional/drugbank/approved_structures_2d.csv",
    "GEOM Drugs": "/homes/buttensc/Projects/semla-flow/data/unconditional/geom-drugs/all.csv",
    ###
    "EQGAT": "predictions/unconditional/eqgat/eqgat_100000_predictions.csv",
    "SemlaFlow": "/homes/buttensc/Projects/semla-flow/predictions/unconditional/semlaflow/semlaflow_100000_predictions.csv",
    "GCDM": "/homes/buttensc/Projects/semla-flow/predictions/unconditional/gcdm/gcdm_100000_predictions.csv",
    "GeoLDM": "/homes/buttensc/Projects/semla-flow/predictions/unconditional/geoldm/geoldm_100000_predictions.csv",
    "MolFlow": "/homes/buttensc/Projects/semla-flow/predictions/unconditional/molflow/molflow_100000_predictions.csv",
}
dfs = []
for name, file in files.items():
    df = pd.read_csv(file, low_memory=False)
    df = df[df[["connected", "chemical"]].all(axis=1)][["smiles"]]  # [:3000]
    df["source"] = name
    dfs.append(df)
df = pd.concat(dfs).reset_index(drop=True)
df = df[df["smiles"].str.len() > 0]
print(len(df))

In [ ]:
# PandasTools.AddMoleculeColumnToFrame(df, "smiles", "mol", includeFingerprints=False)
df["mol"] = df["smiles"].apply(MolFromSmiles)

### Intro

Measuring difference between bit-wise or count-wise fingerprints. 
- Manhatten: L1(f1, f2) = counts how many features are different 



### Define training set

In [ ]:
training_set = df["mol"].apply(lambda x: x is not None)
training_set = df["source"] == "DrugBank"

### Create fingerprint

In [ ]:
ecfp_featuriser = create_sort_and_slice_ecfp_featuriser(
    mols_train=df.loc[training_set, "mol"],
    max_radius=2,
    pharm_atom_invs=False,
    bond_invs=True,
    chirality=False,
    sub_counts=True,
    vec_dimension=1024,  # explore 2048
)

In [ ]:
df["fp"] = df["mol"].apply(ecfp_featuriser)

In [ ]:
X = np.asarray(df.loc[training_set, "fp"].tolist())

# model = PCA(n_components=2)
# model = SparsePCA(n_components=2)

# model = TSNE(n_components=2, perplexity=30, metric="manhattan")

# reducer = UMAP(metric="correlation", min_dist=1.0, spread=1.0)
# reducer = UMAP(metric="correlation", min_dist=0.5, spread=1.0)
# reducer = UMAP(metric="cosine", min_dist=1.0, spread=1.0)
# reducer = UMAP(metric="cosine", min_dist=0.5, spread=1.0)
# model = UMAP(metric="dice", min_dist=1.0, spread=1.0)
model = UMAP(metric="manhattan", min_dist=1.0, spread=1.0)

model.fit(X)

In [ ]:
df[["x", "y"]] = model.transform(np.asarray(df["fp"].tolist()))

In [ ]:
# facetgrid with seaborn scatter, different color each
g = sns.FacetGrid(df, col="source", hue="source", col_wrap=2, despine=False, margin_titles=False)
g.map(sns.scatterplot, "x", "y", s=5, linewidth=0, alpha=1.0)
g.figure.subplots_adjust(wspace=0, hspace=0)
for ax in g.axes.flatten():
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.text(0.03, 0.97, ax.get_title().split("=")[1].strip(), transform=ax.transAxes, ha="left", va="top", fontsize=10, color="black")
    ax.set_title("")
# g.add_legend()
